# 04 - Tracking inventory

**Goal.** List, from the data itself, what the game tracks: every event, its properties, how often each property is filled, and when each one first and last appears. This inventory is the base of `docs/tracking_plan.md`.

**Rule.** Everything here is observed, not documented by the game team. When the plan says *when* an event is triggered, that is an interpretation of the event name and its properties.

**Output.** Three CSV files in `docs/`: `event_inventory.csv`, `event_properties.csv`, `user_properties.csv`. They contain counts, names and a few example values for low-cardinality properties only (no identifiers, no receipts).

In [8]:
import os, re, sys
sys.path.append("../scripts")
from load_events import connect_events
import pandas as pd

con, readable, unreadable = connect_events("../data/raw/player_analytics/*.parquet")
print(len(readable), "readable files,", len(unreadable), "unreadable")
os.makedirs("../docs", exist_ok=True)

67 readable files, 1 unreadable


## 1. Event inventory

One row per event name: volume, players, first and last day seen, and the share of events that have no session id.

In [9]:
df_events = con.execute("""
    SELECT event_name,
           COUNT(*)                                             AS events,
           COUNT(DISTINCT user_id)                              AS players,
           MIN(CAST(ts AS DATE))                                AS first_day,
           MAX(CAST(ts AS DATE))                                AS last_day,
           ROUND(100.0 * COUNT(*) FILTER (WHERE user_session_id = '') / COUNT(*), 1) AS pct_without_session
    FROM events
    GROUP BY 1
    ORDER BY events DESC
""").df()
df_events.to_csv("../docs/event_inventory.csv", index=False)
print(len(df_events), "distinct event names")
df_events.head(25)

66 distinct event names


,event_name,events,players,first_day,last_day,pct_without_session
0,tokenCashout,9425816,64318,2025-02-28,2025-04-10,0.0
1,lostCustomer,6638996,58866,2025-02-28,2025-04-10,0.0
2,interstitialLifecycle,5129605,64613,2025-02-28,2025-04-10,0.0
3,rvLifecycle,4460380,65083,2025-02-28,2025-04-10,0.0
4,appBackground,4207157,65316,2025-02-28,2025-04-10,0.0
5,transferMoney,3890433,63927,2025-02-28,2025-04-10,0.0
6,appFocus,3626170,65542,2025-02-28,2025-04-10,0.0
7,achievementUpdated,3525219,59054,2025-02-28,2025-04-10,100.0
8,deviceBroken,3240686,55712,2025-02-28,2025-04-10,0.0
9,deviceRepair,2961672,53868,2025-02-28,2025-04-10,0.0


Events that start late or stop early point to tracking changes (new version, removed feature) or to the data gap of 2025-03-27.

In [10]:
df_events[(df_events["first_day"].astype(str) > "2025-03-01") | (df_events["last_day"].astype(str) < "2025-04-09")]

,event_name,events,players,first_day,last_day,pct_without_session
23,rvRewardClaim,498847,28327,2025-03-12,2025-04-10,0.0
24,_propertiesUpdate,412485,33495,2025-03-24,2025-04-10,0.0
45,_experimentJoin,3379,3329,2025-03-07,2025-04-10,100.0
49,inAppPurchase,411,264,2025-02-28,2025-03-21,0.0
63,gem6,41,28,2025-03-01,2025-04-08,0.0
64,gem5,33,28,2025-03-02,2025-04-10,0.0


## 2. Naming conventions

Consistent names make the data easier to use. The check below only looks at the form of the names.

In [11]:
def style(name):
    if name.startswith("_"):
        return "system (leading underscore)"
    if re.search(r"\d", name):
        return "contains a digit"
    if re.fullmatch(r"[a-z]+", name):
        return "single lowercase word"
    if re.fullmatch(r"[a-z]+([A-Z][a-z0-9]+)+", name):
        return "camelCase"
    return "other"

df_events["style"] = df_events["event_name"].map(style)
print(df_events.groupby("style")["event_name"].agg(["count", lambda s: ", ".join(s.head(8))]).rename(columns={"<lambda_0>": "examples"}).to_string())

                             count                                                                                                                    examples
style                                                                                                                                                         
camelCase                       52  tokenCashout, lostCustomer, interstitialLifecycle, rvLifecycle, appBackground, transferMoney, appFocus, achievementUpdated
contains a digit                 9                                                   noAds24HoursOffer, gem1, gem2, gem3, robot3Days, gem4, noAds24Hours, gem6
single lowercase word            1                                                                                                                       robot
system (leading underscore)      4                                                          _propertiesUpdate, _sessionStart, _identityCreate, _experimentJoin


Names that contain a digit (`gem1`, `gem2`, ...) look like product names used as event names: each new product would create a new event. A single `purchaseCompleted` event with a product property is cleaner (and already exists).

## 3. Event properties

For each event and each property key: number of events that carry it, the share of that event's rows, the approximate number of distinct values, first and last day seen. An example value is kept only when the property has 20 distinct values or fewer (a category such as `adFormat`); free-text and identifier-like properties get no example. This scans all rows and takes a few minutes.

In [12]:
df_props = con.execute("""
    SELECT event_name,
           k                                   AS property,
           COUNT(*)                            AS n_events,
           APPROX_COUNT_DISTINCT(v)            AS approx_distinct_values,
           MIN(day)                            AS first_day,
           MAX(day)                            AS last_day,
           ANY_VALUE(LEFT(v, 40))              AS example
    FROM (SELECT event_name,
                 CAST(ts AS DATE)                    AS day,
                 UNNEST(map_keys(event_properties))   AS k,
                 UNNEST(map_values(event_properties)) AS v
          FROM events)
    GROUP BY 1, 2
""").df()
df_props = df_props.merge(df_events[["event_name", "events"]], on="event_name")
df_props["pct_of_event"] = (100 * df_props["n_events"] / df_props["events"]).round(1)
df_props.loc[df_props["approx_distinct_values"] > 20, "example"] = ""
df_props = df_props.drop(columns="events").sort_values(["event_name", "n_events"], ascending=[True, False])
df_props.to_csv("../docs/event_properties.csv", index=False)
print(len(df_props), "event/property pairs")
df_props.head(25)

189 event/property pairs


,event_name,property,n_events,approx_distinct_values,first_day,last_day,example,pct_of_event
18,_experimentJoin,experiment_phase_id,3379,1,2025-03-07,2025-04-10,7822f0a0-95bd-44fc-8339-9d88026394c1,100.0
19,_experimentJoin,experiment_phase_name,3379,1,2025-03-07,2025-04-10,AdsConfig_iOS,100.0
93,_experimentJoin,experiment_variant_id,3379,3,2025-03-07,2025-04-10,a2cf25ce-bf6e-468b-94c5-9a5d380cd066,100.0
123,_experimentJoin,experiment_name,3379,1,2025-03-07,2025-04-10,AdsConfig_iOS,100.0
187,_experimentJoin,experiment_id,3379,1,2025-03-07,2025-04-10,544551aa-3fb2-4f7e-bd04-f46ea72777ea,100.0
188,_experimentJoin,experiment_variant_name,3379,3,2025-03-07,2025-04-10,Control-Variant,100.0
103,_identityCreate,identity_id,51743,51913,2025-02-28,2025-04-10,,100.0
23,_sessionStart,identity_id,286686,74573,2025-02-28,2025-04-10,,100.0
56,_sessionStart,session_id,286686,276202,2025-02-28,2025-04-10,,100.0
94,_sessionStart,token_expires_at,286686,259167,2025-02-28,2025-04-10,,100.0


Properties with `pct_of_event` well below 100 are optional or were added later: compare `first_day` with the start of the window (2025-02-28).

In [13]:
df_props[(df_props["pct_of_event"] < 100) & (df_props["n_events"] > 1000)].sort_values("pct_of_event").head(25)

,event_name,property,n_events,approx_distinct_values,first_day,last_day,example,pct_of_event
22,adImpression,amount,578369,468836,2025-03-24,2025-04-10,,30.4
39,adImpression,value,1322491,950436,2025-02-28,2025-04-10,,69.6
65,_sessionStart,refresh_token_expires_at,275178,249098,2025-02-28,2025-04-10,,96.0


## 4. User properties

`user_properties_custom` is attached to every event (player-level attributes such as app version). It is repeated on millions of rows, so a 1% sample is enough to list the keys and how often they are filled.

In [14]:
con.execute("CREATE OR REPLACE TABLE ev_sample AS SELECT user_properties_custom FROM events USING SAMPLE 1 PERCENT")
n_sample = con.execute("SELECT COUNT(*) FROM ev_sample").fetchone()[0]
df_user = con.execute("""
    SELECT k AS property,
           COUNT(*)                         AS n_in_sample,
           APPROX_COUNT_DISTINCT(v)         AS approx_distinct_values,
           ANY_VALUE(LEFT(v, 40))           AS example
    FROM (SELECT UNNEST(map_keys(user_properties_custom))   AS k,
                 UNNEST(map_values(user_properties_custom)) AS v
          FROM ev_sample)
    GROUP BY 1
    ORDER BY 2 DESC
""").df()
df_user["pct_of_events"] = (100 * df_user["n_in_sample"] / n_sample).round(1)
df_user.loc[df_user["approx_distinct_values"] > 20, "example"] = ""
df_user.to_csv("../docs/user_properties.csv", index=False)
print(n_sample, "events in the sample;", len(df_user), "user properties")
df_user

706560 events in the sample; 8 user properties


,property,n_in_sample,approx_distinct_values,example,pct_of_events
0,currentMap,705447,7,1,99.8
1,appVersion,704115,17,0.21.20,99.7
2,fbInstallationId,704115,26672,,99.7
3,robotOwned,704085,2,False,99.6
4,noAdsOwned,704085,2,False,99.6
5,adjustId,703785,25941,,99.6
6,dbChurnPrediction,180365,9428,,25.5
7,dbPurchasePrediction,10447,451,,1.5


A property found on a small share of events cannot be relied on as a player attribute. The sample may miss very rare keys.

## Output

The three CSV files are the base of `docs/tracking_plan.md`. In the plan, each *trigger* and *use* is marked as an interpretation, since the game's code and design documents were not available.